## Optimasi Efisiensi Nutrisi pada Sistem Aeroponik Melalui Pengontrolan Kontinu Pengkabutan dengan Algoritma A2C

In [26]:
!pip install stable-baselines3

In [27]:
# ==========================================
# SECTION 1: IMPORT LIBRARIES
# ==========================================
# Mengimpor pustaka dasar yang diperlukan untuk manipulasi data,
# komputasi numerik, dan pemodelan logika fungsi imbalan.
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from stable_baselines3 import A2C
from stable_baselines3.common.env_checker import check_env

print("Pustaka berhasil diimpor.")

Pustaka berhasil diimpor.


##Definisi Ruang Status ($S(t)$)
Representasi matematis dari sistem pada waktu tertentu ($t$) didefinisikan melalui vektor status $S(t)$

Variabel Biologis & Visual
*   Panjang Akar ($L_{root}(t)$)
*   Kesehatan Umbi ($H_{root}(t)$) Normalisasi (0-1)
*   Status Keberadaan Umbi  ($U_{status}(t)$) Boolean (0/1)

Variabel Lingkungan Mikro (Boks Kultur):
*   Temperatur Internal ($T_{in}(t)$)
*   Kelembapan Internal ($H_{in}(t)$)

Variabel Lingkungan Eksternal & Nutrisi:
*   Temperature Eksternal ($T_{out}(t)$)
*   Kelembaban Eksternal ($H_{out}(t)$)
*   Konduktivitas Elektrik Nutrisi ($EC(t)$)
*   Derajat Keasaman Nutrisi ($pH(t)$)
*   Temperatur Nutrisi ($T_{nut}(t)$)
*   Kondisi Siang dan Malam ($I_{day}(t)$)




In [28]:
# ==========================================
# SECTION 2: DEFINISI RUANG STATUS (State Space S(t))
# ==========================================
class AeroponicStateSpace:
    """
    Kelas untuk merepresentasikan dan memvalidasi vektor status sistem S(t)
    pada sistem budidaya kentang aeroponik.
    """
    def __init__(self, raw_state: Dict[str, float]):
        # Variabel Biologis & Visual
        self.L_root = raw_state.get('L_root', 0.0)         # Panjang Akar (cm atau mm)
        self.U_status = raw_state.get('U_status', 0)       # Status Keberadaan Umbi (Boolean: 0/1)

        # Variabel Lingkungan Mikro (Boks Kultur)
        self.T_in = raw_state.get('T_in', 25.0)            # Temperatur Internal (°C)
        self.H_in = raw_state.get('H_in', 80.0)            # Kelembapan Internal (% RH)

        # Variabel Lingkungan Eksternal & Nutrisi
        self.T_out = raw_state.get('T_out', 28.0)          # Temperatur Eksternal (°C)
        self.H_out = raw_state.get('H_out', 70.0)          # Kelembapan Eksternal (% RH)
        self.EC = raw_state.get('EC', 1.5)                 # Konduktivitas Elektrik (mS/cm)
        self.pH = raw_state.get('pH', 6.0)                 # Derajat Keasaman Nutrisi
        self.T_nut = raw_state.get('T_nut', 22.0)          # Temperatur Nutrisi (°C)
        self.I_day = raw_state.get('I_day', 1)             # Kondisi Siang dan Malam (Boolean: 0/1)

    def to_vector(self) -> np.ndarray:
        """Mengubah dictionary status menjadi array numerik untuk agen RL."""
        return np.array([
            self.L_root, self.U_status, self.T_in, self.H_in,
            self.T_out, self.H_out, self.EC, self.pH, self.T_nut, self.I_day
        ], dtype=np.float32)

## Definisi Ruang Aksi ($A(t)$)
Kontrol aktuator dirancang untuk mengatur distribusi cairan secara terpisah antara siang dan malam juga zona atas dan bawah guna efisiensi penyerapan
*   Misting Utama Atas ($A_{mist}(t)$) ($D_{mist}$) ($I_{mist}$)
*   Valve Penyiraman Bawah ($A_{valve}(t)$) Boolean (0/1)





In [29]:
# ==========================================
# SECTION 3: DEFINISI RUANG AKSI (Action Space A(t))
# ==========================================
class AeroponicActionSpace:
    """
    Kelas untuk merepresentasikan ruang aksi A(t) yang mencakup
    pengendalian misting atas (kontinu) dan valve bawah (diskrit/boolean).
    """
    def __init__(self, d_mist: float, i_mist: float, a_valve: int):
        self.D_mist = float(d_mist)    # Waktu ON Misting Utama Atas (detik)
        self.I_mist = float(i_mist)    # Waktu OFF Misting Utama Atas (menit)
        self.A_valve = int(a_valve)    # Status Valve Penyiraman Bawah (Boolean: 0/1)

    def validate_actions(self) -> bool:
        """Memastikan parameter aksi berada dalam batas fisik yang aman."""
        d_mist_valid = 0.0 <= self.D_mist <= 30.0
        i_mist_valid = 1.0 <= self.I_mist <= 120.0
        valve_valid = self.A_valve in [0, 1]
        return d_mist_valid and i_mist_valid and valve_valid

## Fungsi imbalan ($R(t)$)
*   Komponen Pertumbuhan: Memberikan insentif positif berdasarkan laju pertambahan panjang akar ($\Delta L_{root}$) atau indikator perkembangan positif lainnya.
*   Komponen Efisiensi Sumber Daya: Memberikan penalti terhadap konsumsi energi yang berlebihan akibat durasi misting yang terlalu lama atau penggunaan air yang tidak efisien.
*   Komponen Batasan Lingkungan (Constraint Penalty): Memberikan penalti jika parameter krusial seperti $pH$, $EC$, atau temperatur internal keluar dari rentang optimal fisiologis tanaman kentang.





In [30]:
# ==========================================
# SECTION 4: PERUMUSAN FUNGSI IMBALAN (Reward Function R(t))
# ==========================================
class AeroponicRewardFunction:
    """
    Menghitung total imbalan R(t) berdasarkan komponen pertumbuhan tanaman,
    efisiensi penggunaan sumber daya, dan penalti pelanggaran batas lingkungan.
    """
    def __init__(self, weights: Dict[str, float], optimal_ranges: Dict[str, Tuple[float, float]]):
        self.w_growth = weights.get('growth', 15.0)
        self.w_mist_cost = weights.get('mist_cost', 0.05)
        self.w_valve_cost = weights.get('valve_cost', 0.2)
        self.w_penalty = weights.get('penalty', 2.0)
        self.optimal_ranges = optimal_ranges

    def _calculate_growth_reward(self, current_l_root: float, prev_l_root: float, dt: float) -> float:
        """Komponen Pertumbuhan: Insentif berdasarkan laju pertambahan panjang akar."""
        delta_length = current_l_root - prev_l_root
        growth_rate = max(0.0, delta_length / dt)
        return self.w_growth * growth_rate

    def _calculate_resource_cost(self, action: AeroponicActionSpace) -> float:
        """Komponen Efisiensi Sumber Daya: Penalti konsumsi energi/air dari durasi aktuator."""
        cost = (self.w_mist_cost * action.D_mist) + (self.w_valve_cost * float(action.A_valve))
        return -cost

    def _calculate_environmental_penalty(self, state: AeroponicStateSpace) -> float:
        """Komponen Batasan Lingkungan: Penalti jika parameter keluar dari rentang optimal."""
        state_dict = {
            'pH': state.pH,
            'EC': state.EC,
            'T_in': state.T_in,
            'H_in': state.H_in
        }

        total_penalty = 0.0
        for param, (opt_min, opt_max) in self.optimal_ranges.items():
            val = state_dict.get(param, opt_min)
            if val < opt_min:
                total_penalty += (opt_min - val)
            elif val > opt_max:
                total_penalty += (val - opt_max)

        return - (self.w_penalty * total_penalty)

    def compute_total_reward(self, current_state: AeroponicStateSpace, prev_state: AeroponicStateSpace,
                             action: AeroponicActionSpace, dt: float) -> float:
        """Menggabungkan seluruh komponen menjadi nilai total imbalan R(t)."""
        r_growth = self._calculate_growth_reward(current_state.L_root, prev_state.L_root, dt)
        r_cost = self._calculate_resource_cost(action)
        r_penalty = self._calculate_environmental_penalty(current_state)

        total_reward = r_growth + r_cost + r_penalty
        return float(total_reward)

In [31]:
# ==========================================
# SECTION 5: PENGUJIAN INTEGRASI MODUL
# ==========================================
if __name__ == "__main__":
    # Konfigurasi bobot imbalan dan rentang optimal fisiologis kentang
    weights_config = {
        'growth': 20.0,
        'mist_cost': 0.1,
        'valve_cost': 0.5,
        'penalty': 5.0
    }

    optimal_limits = {
        'pH': (5.5, 6.5),
        'EC': (1.2, 2.0),
        'T_in': (18.0, 25.0),
        'H_in': (75.0, 95.0)
    }

    # Simulasi data state sebelumnya dan saat ini
    prev_data = {'L_root': 45.0}
    curr_data = {
        'L_root': 47.2, 'U_status': 0, 'T_in': 22.0, 'H_in': 85.0,
        'T_out': 27.0, 'H_out': 65.0, 'EC': 1.6, 'pH': 6.0, 'T_nut': 21.5, 'I_day': 1
    }

    # Inisialisasi objek state dan action
    prev_state = AeroponicStateSpace(prev_data)
    current_state = AeroponicStateSpace(curr_data)
    action_agent = AeroponicActionSpace(d_mist=5.0, i_mist=15.0, a_valve=1)

    # Evaluasi fungsi imbalan
    reward_calculator = AeroponicRewardFunction(weights_config, optimal_limits)
    delta_time = 1.0 # satuan waktu diskrit

    total_reward_value = reward_calculator.compute_total_reward(current_state, prev_state, action_agent, delta_time)

    print(f"\n--- HASIL VALIDASI FUNGSI IMBALAN ---")
    print(f"Vektor Status Berhasil Dibuat: {current_state.to_vector().shape}")
    print(f"Validasi Aksi Kontrol: {action_agent.validate_actions()}")
    print(f"Total Imbalan R(t): {total_reward_value:.4f}")


--- HASIL VALIDASI FUNGSI IMBALAN ---
Vektor Status Berhasil Dibuat: (10,)
Validasi Aksi Kontrol: True
Total Imbalan R(t): 43.0000


## Pengembangan Lingkungan Simulasi (Environment Modeling)
Sebelum algoritma diterapkan ke perangkat keras fisik, model interaksi sistem perlu diuji dalam simulator berbasis data historis atau persamaan diferensial untuk memodelkan:

*   Dinamika penurunan kelembapan boks saat aktuator OFF.

*   Respons laju pertumbuhan akar terhadap variasi penyemprotan nutrisi.

*   Fluktuasi suhu dan kelembapan eksternal terhadap kondisi mikroklimat dalam boks.

In [32]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class AeroponicSimulatorEnv(gym.Env):
    """
    Lingkungan simulasi kustom untuk sistem kontrol aeroponik kentang fase awal.
    """
    metadata = {"render_modes": ["human"]}

    def __init__(self):
        super(AeroponicSimulatorEnv, self).__init__()

        # Definisi Ruang Aksi: [D_mist (detik), I_mist (menit), A_valve (0 atau 1)]
        self.action_space = spaces.Box(
            low=np.array([0.0, 1.0, 0.0], dtype=np.float32),
            high=np.array([30.0, 120.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

        # Definisi Ruang Observasi (Vektor Status S(t)): 10 parameter
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 10.0, 20.0, 10.0, 20.0, 0.5, 4.0, 15.0, 0.0], dtype=np.float32),
            high=np.array([500.0, 1.0, 40.0, 100.0, 40.0, 100.0, 3.5, 9.0, 35.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

        self.state = None
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Inisialisasi kondisi awal boks kultur pada fase vegetatif awal
        self.state = np.array([
            10.0,   # L_root: Panjang akar awal (cm)
            0.0,    # U_status: Belum ada umbi
            24.0,   # T_in: Temperatur internal (°C)
            85.0,   # H_in: Kelembapan internal (% RH)
            28.0,   # T_out: Temperatur eksternal (°C)
            70.0,   # H_out: Kelembapan eksternal (% RH)
            1.5,    # EC: Konduktivitas elektrik (mS/cm)
            6.0,    # pH: Derajat keasaman
            22.0,   # T_nut: Temperatur nutrisi (°C)
            1.0     # I_day: Kondisi siang hari
        ], dtype=np.float32)

        return self.state, {}

    def step(self, action):
        d_mist, i_mist, a_valve = action

        # Simpan state sebelumnya untuk kalkulasi reward pertumbuhan
        prev_l_root = self.state[0]

        # --- Model Dinamika Transisi Fisik & Biologis ---
        # 1. Pertumbuhan Akar (Model logistik dengan faktor kelembapan)
        r = 0.02  # Laju pertumbuhan intrinsik
        K = 300.0 # Kapasitas maksimum panjang akar
        moisture_factor = min(1.0, self.state[3] / 90.0)
        delta_l = r * prev_l_root * (1.0 - prev_l_root / K) * moisture_factor * (d_mist / 10.0)
        self.state[0] = min(K, prev_l_root + delta_l)

        # 2. Dinamika Kelembapan Internal (Peluruhan eksponensial)
        decay_rate = 0.05
        target_humidity = 95.0 if d_mist > 0 else self.state[5]
        self.state[3] = target_humidity - (target_humidity - self.state[3]) * np.exp(-decay_rate)

        # Diskritisasi status valve bawah
        discrete_valve = 1 if a_valve >= 0.5 else 0

        # Evaluasi fungsi imbalan R(t)
        reward = self._compute_reward(prev_l_root, self.state[0], d_mist, discrete_valve)

        # Kondisi terminasi episode simulasi
        terminated = False
        truncated = False

        return self.state, reward, terminated, truncated, {}

    def _compute_reward(self, prev_l, curr_l, d_mist, a_valve):
        # Implementasi ringkas fungsi imbalan R(t)
        r_growth = 20.0 * max(0.0, curr_l - prev_l)
        r_cost = - (0.1 * d_mist + 0.5 * float(a_valve))
        return float(r_growth + r_cost)

In [33]:
#Verifikasi Lingkungan Simulasi
def verify_aeroponic_environment():
    print("Memulai Inisialisasi Lingkungan Simulasi...")
    env = AeroponicSimulatorEnv()

    # 1. Uji Fungsi Reset
    initial_state, info = env.reset()
    print("\n--- HASIL UJI RESET ---")
    print(f"Bentuk Vektor Observasi (State Shape): {initial_state.shape}")
    print(f"Nilai Awal Panjang Akar (L_root): {initial_state[0]:.2f} cm")
    print(f"Nilai Awal Kelembapan Internal (H_in): {initial_state[3]:.2f} % RH")

    # 2. Uji Eksekusi Satu Langkah (Single-Step Execution)
    # Definisi aksi sampel: D_mist = 10.0 detik, I_mist = 30.0 menit, A_valve = 1 (Buka)
    sample_action = np.array([10.0, 30.0, 1.0], dtype=np.float32)

    next_state, reward, terminated, truncated, info = env.step(sample_action)

    print("\n--- HASIL UJI STEP PERTAMA ---")
    print(f"Aksi yang Dieksekusi -> D_mist: {sample_action[0]}s, A_valve: {sample_action[2]}")
    print(f"Panjang Akar Setelah Step (L_root): {next_state[0]:.4f} cm")
    print(f"Kelembapan Internal Setelah Step (H_in): {next_state[3]:.4f} % RH")
    print(f"Nilai Imbalan R(t) yang Dihasilkan: {reward:.4f}")
    print(f"Status Terminasi: {terminated}")

    # 3. Uji Simulasi Multi-Langkah (Smoke Test 10 Iterasi)
    print("\n--- UJI STABILITAS MULTI-LANGKAH (10 ITERASI) ---")
    current_state = next_state
    for step_idx in range(1, 11):
        # Simulasi aksi acak yang valid dalam rentang ruang aksi
        random_action = env.action_space.sample()
        current_state, reward, terminated, truncated, _ = env.step(random_action)

        print(f"Iterasi ke-{step_idx:02d} | L_root: {current_state[0]:.4f} cm | H_in: {current_state[3]:.2f}% | Reward: {reward:.4f}") # Ubah [1] ke [3]
        if terminated or truncated:
            print("Simulasi berhenti lebih awal karena mencapai batas kondisi akhir.")
            break

    print("\nVerifikasi Lingkungan Selesai: Simulator berjalan secara stabil dan logis.")

In [34]:
# Menjalankan fungsi verifikasi
if __name__ == "__main__":
    verify_aeroponic_environment()

Memulai Inisialisasi Lingkungan Simulasi...

--- HASIL UJI RESET ---
Bentuk Vektor Observasi (State Shape): (10,)
Nilai Awal Panjang Akar (L_root): 10.00 cm
Nilai Awal Kelembapan Internal (H_in): 85.00 % RH

--- HASIL UJI STEP PERTAMA ---
Aksi yang Dieksekusi -> D_mist: 10.0s, A_valve: 1.0
Panjang Akar Setelah Step (L_root): 10.1826 cm
Kelembapan Internal Setelah Step (H_in): 85.4877 % RH
Nilai Imbalan R(t) yang Dihasilkan: 2.1518
Status Terminasi: False

--- UJI STABILITAS MULTI-LANGKAH (10 ITERASI) ---
Iterasi ke-01 | L_root: 10.7162 cm | H_in: 85.95% | Reward: 7.3174
Iterasi ke-02 | L_root: 11.0154 cm | H_in: 86.39% | Reward: 3.9680
Iterasi ke-03 | L_root: 11.0548 cm | H_in: 86.81% | Reward: 0.0938
Iterasi ke-04 | L_root: 11.5541 cm | H_in: 87.21% | Reward: 7.5556
Iterasi ke-05 | L_root: 12.0591 cm | H_in: 87.59% | Reward: 7.7546
Iterasi ke-06 | L_root: 12.4691 cm | H_in: 87.95% | Reward: 5.8799
Iterasi ke-07 | L_root: 12.8863 cm | H_in: 88.30% | Reward: 6.5573
Iterasi ke-08 | L_roo

In [35]:
# ==========================================
# 1. KELAS LINGKUNGAN SIMULASI KUSTOM
# ==========================================
class AeroponicSimulatorEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self):
        super(AeroponicSimulatorEnv, self).__init__()

        # Ruang Aksi: [D_mist (detik), I_mist (menit), A_valve (0 atau 1)]
        self.action_space = spaces.Box(
            low=np.array([0.0, 1.0, 0.0], dtype=np.float32),
            high=np.array([30.0, 120.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

        # Ruang Observasi (Vektor Status S(t)): 10 parameter
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 10.0, 20.0, 10.0, 20.0, 0.5, 4.0, 15.0, 0.0], dtype=np.float32),
            high=np.array([500.0, 1.0, 40.0, 100.0, 40.0, 100.0, 3.5, 9.0, 35.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

        self.state = None
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([
            10.0,   # L_root: Panjang akar awal (cm)
            0.0,    # U_status: Belum ada umbi
            24.0,   # T_in: Temperatur internal (°C)
            85.0,   # H_in: Kelembapan internal (% RH)
            28.0,   # T_out: Temperatur eksternal (°C)
            70.0,   # H_out: Kelembapan eksternal (% RH)
            1.5,    # EC: Konduktivitas elektrik (mS/cm)
            6.0,    # pH: Derajat keasaman
            22.0,   # T_nut: Temperatur nutrisi (°C)
            1.0     # I_day: Kondisi siang hari
        ], dtype=np.float32)

        return self.state, {}

    def step(self, action):
        d_mist, i_mist, a_valve = action
        prev_l_root = self.state[0]

        # Model Dinamika Transisi Fisik & Biologis
        r = 0.02
        K = 300.0
        moisture_factor = min(1.0, self.state[3] / 90.0)
        delta_l = r * prev_l_root * (1.0 - prev_l_root / K) * moisture_factor * (d_mist / 10.0)
        self.state[0] = min(K, prev_l_root + delta_l)

        decay_rate = 0.05
        target_humidity = 95.0 if d_mist > 0 else self.state[5]
        self.state[3] = target_humidity - (target_humidity - self.state[3]) * np.exp(-decay_rate)

        discrete_valve = 1 if a_valve >= 0.5 else 0
        reward = self._compute_reward(prev_l_root, self.state[0], d_mist, discrete_valve)

        terminated = False
        truncated = False

        return self.state, reward, terminated, truncated, {}

    def _compute_reward(self, prev_l, curr_l, d_mist, a_valve):
        r_growth = 20.0 * max(0.0, curr_l - prev_l)
        r_cost = - (0.1 * d_mist + 0.5 * float(a_valve))
        return float(r_growth + r_cost)


# ==========================================
# 2. PROSES INISIALISASI DAN PELATIHAN A2C
# ==========================================
if __name__ == "__main__":
    print("Inisialisasi Lingkungan Simulasi untuk A2C...")
    env = AeroponicSimulatorEnv()

    # Validasi kesesuaian lingkungan dengan standar Gymnasium
    check_env(env, warn=True)
    print("Validasi lingkungan Gymnasium berhasil dilewati.")

    # Konfigurasi dan instansiasi model A2C menggunakan Multi-Layer Perceptron (MlpPolicy)
    print("Memuat arsitektur model A2C...")
    model = A2C(
        "MlpPolicy",
        env,
        learning_rate=0.0007,
        n_steps=5,
        verbose=1,
        tensorboard_log="./aeroponic_a2c_tensorboard/"
    )

    # Eksekusi pelatihan agen
    total_timesteps = 10000
    print(f"Memulai proses pelatihan agen A2C selama {total_timesteps} langkah waktu...")
    model.learn(total_timesteps=total_timesteps)

    # Penyimpanan bobot model yang telah dilatih
    model_path = "aeroponic_a2c_policy"
    model.save(model_path)
    print(f"Pelatihan selesai. Model berhasil disimpan pada direktori: {model_path}.zip")

Inisialisasi Lingkungan Simulasi untuk A2C...
Validasi lingkungan Gymnasium berhasil dilewati.
Memuat arsitektur model A2C...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/env_checker.py:515: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(


Memulai proses pelatihan agen A2C selama 10000 langkah waktu...
Logging to ./aeroponic_a2c_tensorboard/A2C_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------
| time/                 |          |
|    fps                | 496      |
|    iterations         | 100      |
|    time_elapsed       | 1        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -4.17    |
|    explained_variance | 9.3e-06  |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 57.1     |
|    std                | 0.972    |
|    value_loss         | 331      |
------------------------------------
------------------------------------
| time/                 |          |
|    fps                | 538      |
|    iterations         | 200      |
|    time_elapsed       | 1        |
|    total_timesteps    | 1000     |
| train/                |          |
|    entropy_loss       | -4.16    |
|    explained_variance | 6.32e-06 |
|    learning_rate      | 0.0007   |
|    n_updates          | 199      |
|    policy_loss        | 51.8     |
|

In [36]:
# ==========================================
# 2. PEMUATAN MODEL DAN EVALUASI INFERENSI
# ==========================================
if __name__ == "__main__":
    print("Inisialisasi lingkungan untuk proses evaluasi...")
    env = AeroponicSimulatorEnv()

    # Memuat model A2C yang telah disimpan sebelumnya
    model_path = "aeroponic_a2c_policy.zip"
    print(f"Memuat model dari berkas: {model_path}...")
    try:
        model = A2C.load(model_path)
        print("Model berhasil dimuat ke dalam memori.")
    except FileNotFoundError:
        print(f"Kesalahan: Berkas model {model_path} tidak ditemukan. Harap jalankan skrip pelatihan terlebih dahulu.")
        exit(1)

    # Menjalankan evaluasi inferensi selama beberapa episode
    num_episodes = 3
    max_steps_per_episode = 20

    print(f"\n--- MEMULAI EVALUASI KEBIJAKAN AGEN ({num_episodes} EPISODE) ---")

    for episode in range(1, num_episodes + 1):
        obs, info = env.reset()
        episode_reward = 0.0

        print(f"\nEpisode ke-{episode}")
        print(f"{'Step':<6} | {'Action D_mist':<15} | {'Action Valve':<12} | {'L_root (cm)':<12} | {'H_in (%)':<10} | {'Reward':<10}")
        print("-" * 75)

        for step in range(1, max_steps_per_episode + 1):
            # Prediksi aksi optimal berdasarkan model terlatih (deterministic=True untuk eksploitasi penuh)
            action, _states = model.predict(obs, deterministic=True)

            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward

            d_mist_val = action[0]
            valve_val = 1 if action[2] >= 0.5 else 0
            l_root_val = obs[0]
            h_in_val = obs[3]

            print(f"{step:<6} | {d_mist_val:<15.2f} | {valve_val:<12} | {l_root_val:<12.4f} | {h_in_val:<10.2f} | {reward:<10.4f}")

            if terminated or truncated:
                break

        print(f"Total Imbalan Kumulatif Episode {episode}: {episode_reward:.4f}")

    print("\nEvaluasi inferensi selesai. Agen siap diintegrasikan dengan perangkat keras mikrokontroler di lapangan.")

Inisialisasi lingkungan untuk proses evaluasi...
Memuat model dari berkas: aeroponic_a2c_policy.zip...
Model berhasil dimuat ke dalam memori.

--- MEMULAI EVALUASI KEBIJAKAN AGEN (3 EPISODE) ---

Episode ke-1
Step   | Action D_mist   | Action Valve | L_root (cm)  | H_in (%)   | Reward    
---------------------------------------------------------------------------
1      | 1.51            | 0            | 10.0276      | 85.49      | 0.4009    
2      | 1.51            | 0            | 10.0554      | 85.95      | 0.4054    
3      | 1.51            | 0            | 10.0835      | 86.39      | 0.4098    
4      | 1.51            | 0            | 10.1117      | 86.81      | 0.4140    
5      | 1.51            | 0            | 10.1402      | 87.21      | 0.4181    
6      | 1.51            | 0            | 10.1688      | 87.59      | 0.4220    
7      | 1.51            | 0            | 10.1976      | 87.95      | 0.4258    
8      | 1.51            | 0            | 10.2266      | 88.30     